# Week 4 Pair Programming: Data Leakage Detective

**Goal:** Find and fix data leakage bugs in broken code

**Dataset:** Adult Income (200-row sample + 5000 noise features — a realistic high-dimensional, small-sample scenario)

**Time:** 30-40 minutes

**Scaffolding:** 40% (Diagnostic exercise - you find and fix bugs)

---

## Your Mission

A junior data scientist wrote code to predict income levels. They collected only **200 records** but logged **thousands of extra measurements** (most are noise). The test accuracy comes back at a **perfect 100%** — way too good to be true. Your job:

1. **Run the broken code** and observe the inflated (100%) performance
2. **Find ALL leakage sources** (there are 3-5 bugs)
3. **Fix each bug** one at a time
4. **Compare results** before and after fixes — watch the accuracy **crash from 100% to ~70%**
5. **Implement the Pipeline solution** to prevent future leakage

> 💡 **Key idea:** Not all leakage is equal. Bugs #1-#3 (imputation/encoding/scaling before the split) leak only a tiny amount. **Bug #4 — selecting the "best" 100 of 5000+ features using the full target — is the killer.** With many noise columns and few rows, some noise will randomly correlate with the labels *including the test rows*, and the model memorizes them. That's what inflates accuracy to 100%.

**Work in pairs:** One person drives (types), one navigates (thinks). Switch roles after finding each bug.

---

## Part 1: The Broken Code (Run This First)

This code has **multiple data leakage bugs**. Run it and observe the suspiciously high accuracy.

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("✅ Imports successful")

In [ ]:
# Load Adult Income dataset
data = fetch_openml('adult', version=2, as_frame=True, parser='auto')
X = data.data
y = data.target

# ---------------------------------------------------------------
# Realistic high-dimensional, small-sample scenario.
# A junior data scientist collected only 200 records but logged
# THOUSANDS of extra measurements (sensor readings, derived columns,
# tracking pixels...) that turn out to be pure noise.
# This is common in genomics, IoT, and ad-tech data.
# Most of these 5000 extra features carry NO real signal.
# ---------------------------------------------------------------
N_SAMPLES = 200
N_NOISE = 5000
rng = np.random.RandomState(42)

sample_idx = rng.choice(len(X), N_SAMPLES, replace=False)
X = X.iloc[sample_idx].reset_index(drop=True)
y = y.iloc[sample_idx].reset_index(drop=True)

noise = pd.DataFrame(
    rng.randn(N_SAMPLES, N_NOISE),
    columns=[f'noise_{i}' for i in range(N_NOISE)]
)
X = pd.concat([X, noise], axis=1)

print(f"Dataset shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

In [ ]:
# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Note: the 5000 noise columns are numeric, so they dominate the count.
real_numeric = [c for c in numeric_features if not c.startswith('noise_')]
print(f"Numeric features: {len(numeric_features)} total "
      f"({len(real_numeric)} real + {len(numeric_features) - len(real_numeric)} noise)")
print(f"Real numeric: {real_numeric}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

### 🐛 BROKEN CODE BELOW - Contains Multiple Leakage Bugs

**Your task:** Read through this code carefully. Can you spot the leakage bugs BEFORE running it?

In [ ]:
# ========================================
# BROKEN CODE - DO NOT USE IN PRODUCTION!
# ========================================

# BUG #1: Imputation BEFORE splitting
print("Step 1: Imputing missing values...")
numeric_imputer = SimpleImputer(strategy='mean')
X[numeric_features] = numeric_imputer.fit_transform(X[numeric_features])

categorical_imputer = SimpleImputer(strategy='most_frequent')
X[categorical_features] = categorical_imputer.fit_transform(X[categorical_features])
print("✅ Imputation complete")

# BUG #2: One-hot encoding BEFORE splitting
print("\nStep 2: One-hot encoding categorical features...")
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_encoded = encoder.fit_transform(X[categorical_features])
X_cat_encoded_df = pd.DataFrame(
    X_cat_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X.index
)

# Combine numeric and encoded categorical
X_processed = pd.concat([X[numeric_features], X_cat_encoded_df], axis=1)
print(f"✅ Encoding complete - {X_processed.shape[1]} features")

# BUG #3: Scaling BEFORE splitting
print("\nStep 3: Scaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)
print("✅ Scaling complete")

# BUG #4: Feature selection BEFORE splitting (uses target variable!)
# ⚠️ THE BIG ONE: picking the 100 "best" of 5000+ features using the
#    FULL target lets noise columns that randomly correlate with y
#    (including on the test rows) sneak in. This is the dominant leak.
print("\nStep 4: Selecting best features...")
selector = SelectKBest(f_classif, k=100)
X_selected = selector.fit_transform(X_scaled, y)
print(f"✅ Feature selection complete - {X_selected.shape[1]} features selected")

# NOW we split (way too late!)
print("\nStep 5: Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# Train model
print("\nStep 6: Training model...")
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print("✅ Model trained")

In [ ]:
# Evaluate the broken model
y_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("="*60)
print("BROKEN CODE RESULTS (WITH LEAKAGE)")
print("="*60)
print(f"Test Accuracy: {test_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("="*60)
print("\n⚠️  ~100% accuracy from NOISE features?! That's impossible without leakage.")
print("The model is memorizing noise that was selected using the test labels.")
print("Your job: Find and fix ALL the bugs — then watch this number crash.\n")

---

## Part 2: Bug Identification (Pair Discussion)

**Before looking at hints below, discuss with your partner:**

1. Which steps happened BEFORE the train/test split?
2. For each preprocessing step before the split, does it use information from the test set?
3. How does each bug cause data leakage?

**Write your findings here:**

### Bug #1: Imputation Leakage

**What's wrong:** ____________________________________________________

**Why it's leakage:** ________________________________________________

**How to fix:** _____________________________________________________

---

### Bug #2: Encoding Leakage

**What's wrong:** ____________________________________________________

**Why it's leakage:** ________________________________________________

**How to fix:** _____________________________________________________

---

### Bug #3: Scaling Leakage

**What's wrong:** ____________________________________________________

**Why it's leakage:** ________________________________________________

**How to fix:** _____________________________________________________

---

### Bug #4: Feature Selection Leakage

**What's wrong:** ____________________________________________________

**Why it's leakage:** ________________________________________________

**How to fix:** _____________________________________________________

---

## Part 3: Manual Fix (Without Pipeline)

Now fix the code manually by moving all preprocessing AFTER the split.

**Scaffolding:** 40% - You implement the correct order with hints.

In [ ]:
# Reload fresh data (undo the leaky preprocessing)
data = fetch_openml('adult', version=2, as_frame=True, parser='auto')
X = data.data
y = data.target

# Rebuild the SAME high-dimensional scenario (200 rows + 5000 noise features)
N_SAMPLES = 200
N_NOISE = 5000
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X), N_SAMPLES, replace=False)
X = X.iloc[sample_idx].reset_index(drop=True)
y = y.iloc[sample_idx].reset_index(drop=True)
noise = pd.DataFrame(
    rng.randn(N_SAMPLES, N_NOISE),
    columns=[f'noise_{i}' for i in range(N_NOISE)]
)
X = pd.concat([X, noise], axis=1)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("✅ Fresh data loaded")

In [ ]:
# TODO: Step 1 - Split data FIRST (before any preprocessing)
# Hint: Use train_test_split on RAW data X, y
# Expected: X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = ____________________

print(f"✅ Data split - Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
# TODO: Step 2 - Impute missing values (FIT on train, TRANSFORM on both)
# Hint: fit_transform on X_train, transform on X_test
# Expected: No test data used during fit()

# Numeric imputation
numeric_imputer = SimpleImputer(strategy='mean')
X_train_num = ______  # Hint: fit_transform on X_train[numeric_features]
X_test_num = ______   # Hint: transform (NOT fit_transform) on X_test[numeric_features]

# Categorical imputation
categorical_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat = ______  # Hint: fit_transform on X_train[categorical_features]
X_test_cat = ______   # Hint: transform on X_test[categorical_features]

print("✅ Imputation complete (no leakage)")

In [ ]:
# TODO: Step 3 - One-hot encode categorical features (FIT on train, TRANSFORM on both)
# Hint: Same pattern - fit_transform on train, transform on test
# Expected: Encoder learns categories from training data only

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat_encoded = ______  # Hint: fit_transform on X_train_cat
X_test_cat_encoded = ______   # Hint: transform on X_test_cat

# Convert to DataFrames
X_train_cat_df = pd.DataFrame(
    X_train_cat_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)
X_test_cat_df = pd.DataFrame(
    X_test_cat_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)

print("✅ Encoding complete (no leakage)")

In [ ]:
# Combine numeric and categorical features
X_train_processed = pd.concat(
    [pd.DataFrame(X_train_num, columns=numeric_features, index=X_train.index),
     X_train_cat_df],
    axis=1
)

X_test_processed = pd.concat(
    [pd.DataFrame(X_test_num, columns=numeric_features, index=X_test.index),
     X_test_cat_df],
    axis=1
)

print(f"✅ Features combined - {X_train_processed.shape[1]} features")

In [ ]:
# TODO: Step 4 - Scale features (FIT on train, TRANSFORM on both)
# Hint: Same pattern as before
# Expected: Scaler learns mean/std from training data only

scaler = StandardScaler()
X_train_scaled = ______  # Hint: fit_transform on X_train_processed
X_test_scaled = ______   # Hint: transform on X_test_processed

print("✅ Scaling complete (no leakage)")

In [ ]:
# TODO: Step 5 - Feature selection (FIT on train, TRANSFORM on both)
# Hint: Selector uses y_train (NOT y_test!)
# Expected: Features selected based on TRAINING data only
# This is the fix for the BIG leak - noise features can no longer
# "peek" at the test labels, so the inflated accuracy disappears.

selector = SelectKBest(f_classif, k=100)
X_train_selected = ______  # Hint: fit_transform on X_train_scaled with y_train
X_test_selected = ______   # Hint: transform on X_test_scaled

print(f"✅ Feature selection complete - {X_train_selected.shape[1]} features (no leakage)")

In [ ]:
# Train model on leak-free data
model_fixed = LogisticRegression(max_iter=1000, random_state=42)
model_fixed.fit(X_train_selected, y_train)

# Evaluate
y_pred_fixed = model_fixed.predict(X_test_selected)
test_accuracy_fixed = accuracy_score(y_test, y_pred_fixed)

print("="*60)
print("FIXED CODE RESULTS (NO LEAKAGE)")
print("="*60)
print(f"Test Accuracy: {test_accuracy_fixed:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_fixed))
print("="*60)

---

## Part 4: Compare Results

Compare the broken (leaky) version with the fixed version.

In [ ]:
print("="*60)
print("COMPARISON: Leaky vs Fixed Code")
print("="*60)
print(f"\nBroken Code (WITH leakage):  {test_accuracy:.3f}")
print(f"Fixed Code (NO leakage):      {test_accuracy_fixed:.3f}")
print(f"\nDifference: {(test_accuracy - test_accuracy_fixed)*100:.1f} percentage points")
print("="*60)
print("\n💡 Key Insight: Data leakage inflated accuracy by exposing")
print("   test set information during preprocessing!\n")

### Discussion Questions

**Answer with your partner:**

1. **Why did the broken code have higher accuracy?**

   _Your answer:_ ___________________________________________________________

2. **Which bug caused the MOST leakage? Why?**

   _Your answer:_ ___________________________________________________________

3. **How would you detect leakage in a real project?**

   _Your answer:_ ___________________________________________________________

4. **Why is manual preprocessing (like we just did) error-prone?**

   _Your answer:_ ___________________________________________________________

---

## Part 5: The Pipeline Solution

Manual preprocessing is tedious and error-prone. Now implement the same workflow using sklearn Pipeline - the leakage-proof solution.

**Scaffolding:** 40% - You build the Pipeline with hints.

In [ ]:
# Reload fresh data again
data = fetch_openml('adult', version=2, as_frame=True, parser='auto')
X = data.data
y = data.target

# Rebuild the SAME high-dimensional scenario (200 rows + 5000 noise features)
N_SAMPLES = 200
N_NOISE = 5000
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X), N_SAMPLES, replace=False)
X = X.iloc[sample_idx].reset_index(drop=True)
y = y.iloc[sample_idx].reset_index(drop=True)
noise = pd.DataFrame(
    rng.randn(N_SAMPLES, N_NOISE),
    columns=[f'noise_{i}' for i in range(N_NOISE)]
)
X = pd.concat([X, noise], axis=1)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Split FIRST (Pipeline doesn't change this step)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✅ Fresh data loaded and split")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: Step 1 - Create numeric preprocessing pipeline
# Hint: Pipeline with SimpleImputer + StandardScaler
# Expected: Two-step pipeline for numeric features

numeric_transformer = Pipeline([
    ('imputer', ______),  # Hint: SimpleImputer(strategy='mean')
    ('scaler', ______)    # Hint: StandardScaler()
])

print("✅ Numeric transformer created")

In [ ]:
# TODO: Step 2 - Create categorical preprocessing pipeline
# Hint: Pipeline with SimpleImputer + OneHotEncoder
# Expected: Two-step pipeline for categorical features

categorical_transformer = Pipeline([
    ('imputer', ______),  # Hint: SimpleImputer(strategy='most_frequent')
    ('onehot', ______)    # Hint: OneHotEncoder(handle_unknown='ignore')
])

print("✅ Categorical transformer created")

In [ ]:
# TODO: Step 3 - Combine transformers with ColumnTransformer
# Hint: ColumnTransformer takes list of (name, transformer, columns) tuples
# Expected: Preprocessor that handles numeric and categorical separately

preprocessor = ColumnTransformer([
    ('num', ______, ______),  # Hint: numeric_transformer, numeric_features
    ('cat', ______, ______)   # Hint: categorical_transformer, categorical_features
])

print("✅ ColumnTransformer created")

In [ ]:
# TODO: Step 4 - Create full pipeline with preprocessor + feature selection + classifier
# Hint: Pipeline with 3 steps: preprocessor, selector, classifier
# Expected: Complete pipeline that prevents all leakage

full_pipeline = Pipeline([
    ('preprocessor', ______),  # Hint: Use the preprocessor from above
    ('selector', ______),      # Hint: SelectKBest(f_classif, k=100)
    ('classifier', ______)     # Hint: LogisticRegression(max_iter=1000, random_state=42)
])

print("✅ Full pipeline created")
print("\nPipeline steps:")
for step_name, _ in full_pipeline.steps:
    print(f"  - {step_name}")

In [ ]:
# Fit pipeline (automatically handles fit_transform on train, transform on test)
full_pipeline.fit(X_train, y_train)
print("✅ Pipeline fitted")

In [ ]:
# Evaluate pipeline
y_pred_pipeline = full_pipeline.predict(X_test)
pipeline_accuracy = accuracy_score(y_test, y_pred_pipeline)

print("="*60)
print("PIPELINE RESULTS (LEAKAGE-PROOF)")
print("="*60)
print(f"Test Accuracy: {pipeline_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_pipeline))
print("="*60)

In [ ]:
# Verify pipeline matches manual fix
print("="*60)
print("VERIFICATION: Manual Fix vs Pipeline")
print("="*60)
print(f"\nManual fix accuracy:  {test_accuracy_fixed:.3f}")
print(f"Pipeline accuracy:    {pipeline_accuracy:.3f}")
print(f"\nDifference: {abs(test_accuracy_fixed - pipeline_accuracy):.4f}")
print("="*60)
print("\n✅ Pipeline gives same results as manual fix!")
print("   But Pipeline is easier, safer, and prevents leakage automatically.\n")

---

## Part 6: Cross-Validation with Pipeline

Now use cross-validation with your pipeline to get a robust performance estimate.

In [ ]:
# TODO: Perform 5-fold cross-validation on the pipeline
# Hint: Use cross_val_score with the full_pipeline
# Expected: Array of 5 CV scores

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = ______  # Hint: cross_val_score(full_pipeline, X_train, y_train, cv=cv)

print("="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
print(f"\nFold scores: {cv_scores}")
print(f"\nMean CV Score: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"Test Score:    {pipeline_accuracy:.3f}")
print("="*60)
print("\n💡 CV score should be close to test score (no leakage!)\n")

---

## Part 7: Save Your Pipeline

Finally, save your leakage-proof pipeline for future use.

In [ ]:
# TODO: Save the pipeline using joblib
# Hint: joblib.dump(model, 'filename.pkl')
# Expected: Pipeline saved to disk

import joblib

______  # Hint: joblib.dump(full_pipeline, 'adult_income_pipeline.pkl')

print("✅ Pipeline saved to 'adult_income_pipeline.pkl'")

In [ ]:
# TODO: Load and test the saved pipeline
# Hint: joblib.load('filename.pkl')
# Expected: Loaded pipeline works exactly like original

loaded_pipeline = ______  # Hint: joblib.load('adult_income_pipeline.pkl')

# Test on a few samples
test_predictions = loaded_pipeline.predict(X_test[:5])
print(f"Loaded pipeline predictions: {test_predictions}")
print(f"Original predictions:        {y_pred_pipeline[:5]}")
print("\n✅ Loaded pipeline works correctly!")

---

## Summary & Reflection

### What You Accomplished

✅ Identified 4+ data leakage bugs in broken code

✅ Fixed leakage manually using correct fit/transform order

✅ Observed accuracy drop when leakage was removed (this is GOOD!)

✅ Built leakage-proof Pipeline with ColumnTransformer

✅ Verified Pipeline matches manual fix

✅ Used cross-validation to validate no leakage

✅ Saved production-ready pipeline

---

### Key Takeaways (Write 3-5 sentences)

**What did you learn about data leakage?**

_Your answer:_



**Why is Pipeline better than manual preprocessing?**

_Your answer:_



**How will you prevent leakage in future projects?**

_Your answer:_



---

### Leakage Prevention Checklist (For Future Projects)

Before deploying any model, verify:

- [ ] Split data BEFORE any preprocessing
- [ ] All preprocessing in Pipeline (no manual fit/transform)
- [ ] Used fit_transform ONLY on training data
- [ ] Used transform (not fit_transform) on test data
- [ ] Feature selection uses training labels only
- [ ] Cross-validation score close to test score
- [ ] No suspiciously high accuracy (>95% is often leakage)
- [ ] Pipeline saved and tested on new data

---

## 🎉 Congratulations!

You're now a **Data Leakage Detective**! You can:
- Spot leakage bugs in code
- Fix leakage manually
- Build leakage-proof Pipelines
- Validate models using cross-validation

**This skill separates hobbyists from professionals. Use it wisely!**

---

*Next: Apply these techniques to your post-class exercise on a new dataset.*